In [1]:
import faiss
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

In [2]:
from sklearn.neighbors import NearestNeighbors

In [3]:

def standardize_data(data: np.ndarray):
    if not np.issubdtype(data.dtype, np.floating):
        data = data.astype(np.float32, copy=False)

    means = np.mean(data, axis=0, dtype=np.float32)
    stds = np.std(data, axis=0, dtype=np.float32)

    stds[stds == 0] = 1.0

    data -= means
    data /= stds

    return data


def apply_pca(data, n_components=50):
    pca = PCA(n_components=n_components)
    reduced = pca.fit_transform(data)
    return reduced


In [16]:
def sklearn_neigh(data, n_neighbors):
    neigh = NearestNeighbors(n_neighbors=n_neighbors, n_jobs=-1)
    nbrs = neigh.fit(data)

    distances, x = nbrs.kneighbors(data)
    return distances, x

In [17]:
def faiss_neigh(data, n_neighbors):
    data = np.ascontiguousarray(data.astype('float32'))

    # Строим индекс Faiss (точный, L2)
    index = faiss.IndexFlatL2(data.shape[1])
    index.add(data)

    # Ищем n_neighbors ближайших точек (включая саму точку — она будет первой с расстоянием 0)
    distances, x = index.search(data, n_neighbors)
    return distances, x

In [6]:
emb = pd.read_parquet('/projects/immunestatus/rheum/tcremp/as_Shep_SFCD8_embeddings.parquet')

In [7]:
emb

,0_b_v,0_b_j,0_b_cdr3,1_b_v,1_b_j,1_b_cdr3,2_b_v,2_b_j,2_b_cdr3,3_b_v,...,2996_b_cdr3,2997_b_v,2997_b_j,2997_b_cdr3,2998_b_v,2998_b_j,2998_b_cdr3,2999_b_v,2999_b_j,2999_b_cdr3
0,770,223,1240,806,198,810,750,223,840,750,...,610,635,198,830,622,197,1090,738,197,620
1,376,196,1470,0,107,820,768,196,970,768,...,860,689,107,560,712,0,1280,492,0,870
2,861,205,1500,891,184,790,911,205,1200,911,...,930,878,184,770,877,183,1250,833,183,640
3,861,0,1130,891,221,880,911,0,590,911,...,1100,878,221,880,877,196,1220,833,196,570
4,751,221,1370,775,0,940,703,221,1290,703,...,1340,560,0,980,627,107,1280,771,107,850
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17532,814,221,1490,814,0,640,820,221,1130,820,...,1180,829,0,700,804,107,1260,776,107,790
17533,814,196,1490,814,107,700,820,196,1250,820,...,1120,829,107,780,804,0,1040,776,0,730
17534,776,217,1070,772,204,1080,700,217,910,700,...,880,585,204,680,574,189,1320,752,189,910
17535,776,196,1530,772,107,1020,700,196,1110,700,...,1300,585,107,900,574,0,1260,752,0,1090


In [8]:
emb = standardize_data(emb.values)

In [9]:
emb = apply_pca(emb)

In [11]:
emb.shape

(17537, 50)

In [18]:
dist_sklearn, xs = sklearn_neigh(emb, n_neighbors=4)

In [19]:
dist_faiss, xf = faiss_neigh(emb, n_neighbors=4)

In [22]:
xf == xs

array([[ True,  True,  True,  True],
       [ True,  True,  True,  True],
       [ True,  True, False, False],
       ...,
       [ True,  True,  True,  True],
       [ True,  True,  True,  True],
       [ True,  True,  True,  True]])

In [23]:
xs

array([[    0,  1433, 14091, 14090],
       [    1, 16134, 16110,  3515],
       [    2,  9808, 10047,  1897],
       ...,
       [ 4567, 17534, 12729, 12435],
       [14634, 17535,  6635,  3448],
       [  500, 17536,  2730,  2288]])

In [20]:
xf

array([[    0,  1433, 14091, 14090],
       [    1, 16134, 16110,  3515],
       [    2,  9808,  1897, 10047],
       ...,
       [ 4567, 17534, 12729, 12435],
       [14634, 17535,  6635,  3448],
       [  500, 17536,  2730,  2288]])

In [24]:
np.sqrt(dist_faiss)

array([[ 0.        , 11.161195  , 12.993011  , 13.218417  ],
       [ 0.05412659, 23.240463  , 28.372677  , 28.658226  ],
       [ 0.        , 13.920393  , 19.320234  , 19.320234  ],
       ...,
       [ 0.0625    ,  0.0625    , 22.953417  , 23.050217  ],
       [ 0.04419417,  0.04419417, 32.767014  , 35.008015  ],
       [ 0.        ,  0.        , 16.110823  , 17.145773  ]],
      dtype=float32)

In [15]:
dist_sklearn

array([[1.90734863e-06, 1.11610861e+01, 1.29929676e+01, 1.32183571e+01],
       [0.00000000e+00, 2.32404804e+01, 2.83726101e+01, 2.86582413e+01],
       [4.26496126e-06, 1.39205618e+01, 1.93204403e+01, 1.93204403e+01],
       ...,
       [0.00000000e+00, 0.00000000e+00, 2.29533348e+01, 2.30501461e+01],
       [2.33601554e-06, 2.33601554e-06, 3.27670326e+01, 3.50080261e+01],
       [1.34869913e-06, 1.34869913e-06, 1.61108704e+01, 1.71457748e+01]])

In [25]:
df = pd.read_parquet('../../tcremp/distances.parquet')

In [26]:
df

,n_0,n_1,n_2,n_3,n_4,n_5,n_6,n_7,n_8,n_9,...,n_40,n_41,n_42,n_43,n_44,n_45,n_46,n_47,n_48,n_49
0,0.044194,13.297123,13.297123,15.068982,17.515924,20.605743,20.776855,21.587757,21.713039,22.059401,...,27.739811,27.799629,27.948141,27.967144,28.191414,28.295336,28.348366,28.411354,28.487936,28.689322
1,0.000000,12.923377,13.393679,13.402280,14.644218,15.371125,16.067120,16.246754,16.360502,16.377148,...,20.920843,20.945520,21.064817,21.202751,21.302601,21.384546,21.446062,21.511715,21.642651,21.655552
2,0.031250,7.734612,18.468115,20.371214,24.020315,24.045570,24.085520,24.657854,24.804115,25.185949,...,31.316570,31.507439,31.514366,31.539101,31.754368,31.779192,32.030334,32.197117,32.254875,32.258827
3,0.062500,21.726055,24.732199,26.661762,27.920017,29.042332,29.173845,29.564713,30.444525,31.122679,...,38.938618,38.985435,38.989544,39.026409,39.035240,39.102692,39.121906,39.349197,39.350166,39.353527
4,0.000000,11.437671,11.723790,11.974827,12.387304,13.286360,14.587024,15.581314,16.696335,17.081606,...,23.049623,23.203709,23.333519,23.421745,23.434208,23.480793,23.541727,23.688160,23.774452,23.850281
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
312046,0.062500,18.847507,22.675531,24.343328,26.944859,27.397251,27.521688,28.099676,28.937229,29.537682,...,35.209946,35.301933,35.334202,35.397549,35.455437,35.474792,35.580921,35.620174,35.734810,35.846569
312047,0.031250,16.919415,18.852507,21.147778,21.401917,21.846834,22.148050,22.737440,23.346176,23.436543,...,26.588995,26.642269,26.646961,26.648390,26.733475,26.745016,26.749872,26.773428,26.835455,26.845606
312048,0.000000,14.901565,15.668440,16.552473,16.778660,17.293480,17.679106,17.695173,18.094370,18.536072,...,25.749279,25.856537,25.951578,26.050020,26.187090,26.301178,26.642967,26.671101,26.962358,27.010487
312049,0.000000,13.933120,14.293280,15.372586,17.946262,18.382359,19.138527,19.612953,19.682117,19.827307,...,28.201164,28.276917,28.462301,28.598122,28.882254,28.905371,29.159533,29.223679,29.600893,29.647144
